In [1]:
from torchvision.datasets import MNIST
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import numpy as np
import math


from helpers import *
from loss_and_eval import *




In [ ]:
class Layer:
    def __init__(self, input_neurons, output_neurons, softmax = False):
        self.j = output_neurons
        self.k = input_neurons

        self.W = np.random.normal(scale = 1/math.sqrt(self.j), size = (self.j, self.k)).tolist()
        self.b = np.random.normal(size = (self.j, 1)).tolist()

        self.softmax = softmax

        self.clear_grad()

    def __call__(self, prev_activations):
        return self.forward(prev_activations)

    def forward(self, prev_activations):
        z = matrix_add(matrix_mult(self.W, prev_activations), self.b)
        if self.softmax:
            return vector_softmax(z), z

        return vector_sigmoid(z), z

    def clear_grad(self):
        self.W_grad = [[0.0 for _ in range(self.k)] for _ in range(self.j)]
        self.b_grad = [[0.0] for _ in range(self.j)]



class Convolution:
    def __init__(self, filters, channels, kernel_size, input_size, stride = 1):
        self.filters = filters
        self.channels = channels
        self.kernel_size = kernel_size
        self.input_size = input_size
        self.stride = stride

        self.output_size = (self.input_size - self.kernel_size) // self.stride + 1

 
        self.W = np.random.normal(
            size = (self.filters, self.channels, self.kernel_size, self.kernel_size)
        )
        self.b = np.random.normal(
            size = self.filters
        )

        
        self.dB = np.zeros(shape = self.filters)
        self.dW = np.zeros(shape = (self.filters, self.channels, self.kernel_size, self.kernel_size))

        

    def __call__(self, x):
        return self.forward(x)

    def forward(self, x):
        # x: (channels, length, width)
        
        f_maps = np.zeros(shape=(self.filters, self.output_size, self.output_size))
        weighted_inputs = np.zeros_like(f_maps)

        # hard-coded
        for f in range(self.filters):
            f_map = np.zeros(shape=(self.output_size, self.output_size))
            weighted_input = np.zeros_like(f_map)

            for i in range(self.output_size):
                for j in range(self.output_size):
                    z = self.b[f]

                    for u in range(self.kernel_size):
                        for v in range(self.kernel_size):
                            for c in range(self.channels):
                                input_channel = x[c]
                                W = self.W[f, c, :, :] # (kernel_size, kernel_size)

                                row = self.stride * i + u
                                col = self.stride * j + v
                                z += input_channel[row][col] * W[u][v]


                    f_map[i][j] = sigmoid(z)
                    weighted_input[i][j] = z

            f_maps[f] = f_map
            weighted_inputs[f] = weighted_input

        return f_maps, weighted_inputs

        # matrix implementation


    def backward(self, dA, prev_activations, weighted_inputs):

        dZ = np.zeros(shape = (self.filters, self.output_size, self.output_size))
        prev_dA = np.zeros(shape = (self.channels, self.input_size, self.input_size))



        for f in range(self.filters):
            for i in range(self.output_size):
                for j in range(self.output_size):
                    dZ[f][i][j] = dsigmoid(weighted_inputs[f][i][j]) * dA[f][i][j]
                    self.dB[f] += dZ[f][i][j]

                    for u in range(self.kernel_size):
                        for v in range(self.kernel_size):
                            for c in range(self.channels): 
                                W = self.W[f, c, :, :]
                                row = self.stride * i + u
                                col = self.stride * j + v


                                self.dW[f][c][u][v] += dZ[f][i][j] * prev_activations[c][row][col]
                                prev_dA[c][row][col] += dZ[f][i][j] * W[f][c][u][v]


        return prev_dA




        
                            


         


class MaxPool:
    def __init__(self, f_maps, input_size, pool_size):
        self.f_maps = f_maps
        self.input_size = input_size
        self.pool_size = pool_size

        assert self.input_size % self.pool_size == 0, f"Pool size {self.pool_size} is not valid for {self.input_size}x{self.input_size} feature maps"
        self.output_size = self.input_size // self.pool_size

        self.dZ_prev_dA = np.zeros(shape = (self.f_maps, self.input_size, self.input_size))


    def __call__(self, f_maps):
        return self.forward(f_maps)

    def forward(self, f_maps):
        pooled_f_maps = np.zeros(shape = (self.f_maps, self.output_size, self.output_size))

        for f in range(self.f_maps):
            f_map = f_maps[f]
            pooled_f_map = np.zeros(shape = (self.output_size, self.output_size))

            for i in range(self.output_size):
                for j in range(self.output_size):

                    a = float("-inf")
                    max_i, max_j = 0

                    for u in range(self.pool_size):
                        for v in range(self.pool_size):
                            row = i * self.pool_size + u
                            col = j * self.pool_size + v

                            if f_map[row][col] > a:
                                a = f_map[row][col]
                                max_i, max_j = row, col

                    pooled_f_map[i][j] = a
                    self.dZ_prev_dA[f][max_i][max_j] = 1.0
                    


            pooled_f_maps[f] = pooled_f_map

        return pooled_f_maps, self.dZ_prev_dA

    

    def backward(self, next_W_T, next_dZ):
        dZ = np.reshape(next_W_T @ next_dZ, (self.f_maps, self.output_size, self.output_size))
        prev_dA = np.zeros(shape = (self.f_maps, self.input_size, self.input_size))

        for f in range(self.f_maps):
            for i in range(self.output_size):
                for j in range(self.output_size):
                    for u in range(self.pool_size):
                        for v in range(self.pool_size):
                            row = i * self.pool_size + u
                            col = j * self.pool_size + v

                            prev_dA[f][row][col] = dZ[f][i][j] * self.dZ_prev_dA[f][row][col]


        return prev_dA


        
                    












IndentationError: unindent does not match any outer indentation level (<string>, line 56)